In [39]:
%pip install transformers torch numpy pandas scikit-learn matplotlib torch datasets

Note: you may need to restart the kernel to use updated packages.


In [40]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from transformers import DataCollatorWithPadding

In [41]:
dataset_dict = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

In [42]:
model_path = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_path)
id2label = {0: "safe", 1: "Not safe"}
label2id = {"Safe": 0, "Not Safe": 1}
model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                           num_labels=2,
                                                           id2label=id2label,)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4531.13it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

In [64]:
for name, param in model.named_parameters():
    if "classifier" in name or "pooler" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [65]:
print(dataset_dict)
print(dataset_dict["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
['flags', 'instruction', 'category', 'intent', 'response']


In [73]:
LABEL_COLUMN = "intent"  

# Ensure 0-based integer indexing
unique_labels = sorted(set(dataset[LABEL_COLUMN] for dataset in [dataset_dict["train"]]))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

def preprocess_function(examples):
    # 1. Tokenize input text
    tokenized = tokenizer(examples["response"], truncation=True, max_length=512)
    
    # 2. Add the 'labels' key (Trainer specifically expects this exact name)
    tokenized["labels"] = [label2id[label] for label in examples[LABEL_COLUMN]]
    return tokenized

TypeError: unhashable type: 'Column'

In [68]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [69]:
# defining evaluation metrics
import evaluate


accuracy = evaluate.load("accuracy")
auc_store = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probabilities = np.exp(predictions) /np.exp(predictions).sum(-1, keepdims=True)
    positive_class_probs = probabilities[:, 1]
    auc = np.round(auc_store.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'],3)
    predicted_classes = np.argmax(predictions, axis=1)
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'], 3)

    return {"Accuracy": acc, "AUC": auc}

In [70]:
# Training parameters
lr = 2e-4
batch_size = 8
num_epochs = 10

training_args = TrainingArguments(
    output_dir="bert-phishing-classifier_teacher",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,    
)

In [ ]:
# 1. Build a strict 0-indexed map from your unique labels
unique_labels = sorted(set(dataset_dict["train"]["intent"]))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

# 2. RE-INITIALIZE THE MODEL with the matching num_labels
# (If unique_labels has 11 items, num_labels will be 11, allowing targets 0..10)
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=len(unique_labels),
    label2id=label2id,
    id2label=id2label
)

# 3. Preprocess the text data
def preprocess_function(examples):
    tokenized = tokenizer(examples["response"], truncation=True, max_length=512)
    tokenized["labels"] = [label2id[label] for label in examples["intent"]]
    return tokenized

tokenized_data = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

# 4. Split and pass the updated model to Trainer
split_dataset = tokenized_data["train"].train_test_split(test_size=0.2, seed=42)

trainer = Trainer(
    model=model,  # Re-initialized model instance
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/home/clencyc/Dev/ATTACHMENT/AimSoft-Feedback-Analysis/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


IndexError: Target 10 is out of bounds.

In [ ]:
import torch 
import torch.nn.functional as F

def test_sentiment(text, model, tokenizer, device, max_len=4):
    model.eval()
    tokens = tokenizer(
        text, 
        max_length=max_len, 
        padding="max_length", 
        truncation=True, 
        return_tensors="pt"
    )
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_class].item()

    label_map = {1: "Negative", 0: "Positive"}
    print(f"Text: '{text}")
    print(f"Predicted Sentiment: {label_map[pred_class]} with confidence {confidence:.4f}")
    print(f"Probabilities: Negative: {probs[0][0]:.4f}, Positive: {probs[0][1]:.4f}")
    print("-" * 60)


In [ ]:
test_sentiment("I love this product!", model, tokenizer, device)
test_sentiment("This is the worst experience I've ever had.", model, tokenizer, device)
test_sentiment("The service was okay, not great but not terrible.", model, tokenizer, device)

Text: 'I love this product!
Predicted Sentiment: Positive with confidence 0.5164
Probabilities: Negative: 0.5164, Positive: 0.4836
------------------------------------------------------------
Text: 'This is the worst experience I've ever had.
Predicted Sentiment: Positive with confidence 0.5304
Probabilities: Negative: 0.5304, Positive: 0.4696
------------------------------------------------------------
Text: 'The service was okay, not great but not terrible.
Predicted Sentiment: Positive with confidence 0.5215
Probabilities: Negative: 0.5215, Positive: 0.4785
------------------------------------------------------------


In [ ]:
# Assuming self.bert inside your class is the Hugging Face model
model.bert.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


('./sentiment_model/tokenizer_config.json', './sentiment_model/tokenizer.json')